# 通用训练过程

In [2]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report, roc_curve, auc
import xgboost as xgb
import matplotlib.pyplot as plt
import seaborn as sns
from autogluon.tabular import TabularPredictor
import argparse
import sys
# 增加递归深度限制
sys.setrecursionlimit(1000000)

/home/gxj/anaconda3/envs/mt5/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# 1. 数据读取

In [ ]:
def load_data(file_path):
    data = pd.read_parquet(file_path, engine='fastparquet')
    return data

# 2. 数据预处理

In [ ]:
def preprocess_data(data):
    X = data.iloc[:, :-1]
    y = data.iloc[:, -1]
    X.fillna(X.mean(), inplace=True)
    return X, y

# 3. 处理正负样本数量差距过大

In [ ]:
def handle_imbalanced_data(y):
    pos_weight = (y == 0).sum() / (y == 1).sum()
    return pos_weight

# 4. 特征工程

In [ ]:
def feature_engineering(X):
    # 标准化特征
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)
    
    # 添加多项式特征
    poly = PolynomialFeatures(degree=2, interaction_only=True, include_bias=False)
    X_poly = poly.fit_transform(X_scaled)
    
    return X_poly

# 5. 使用AutoGluon进行超参数调优和模型训练

In [ ]:
def train_with_autogluon(X_train, y_train, X_test, y_test, update_status):
    train_data = pd.DataFrame(X_train)
    train_data['label'] = y_train
    
    test_data = pd.DataFrame(X_test)
    test_data['label'] = y_test
    
    update_status("Starting model training...")
    predictor = TabularPredictor(label='label').fit(train_data, presets='best_quality')
    update_status("Model training completed.")
    
    return predictor

# 6. 模型评估

In [ ]:
def evaluate_model(predictor, X_test, y_test, update_status):
    test_data = pd.DataFrame(X_test)
    test_data['label'] = y_test
    
    y_pred = predictor.predict(test_data.drop(columns=['label']))
    y_pred_proba = predictor.predict_proba(test_data.drop(columns=['label']))[:, 1]
    
    accuracy = accuracy_score(y_test, y_pred)
    conf_matrix = confusion_matrix(y_test, y_pred)
    class_report = classification_report(y_test, y_pred)
    
    update_status(f"Accuracy: {accuracy:.4f}")
    update_status("Confusion Matrix:")
    update_status(str(conf_matrix))
    update_status("Classification Report:")
    update_status(class_report)
    
    # ROC曲线
    fpr, tpr, _ = roc_curve(y_test, y_pred_proba)
    roc_auc = auc(fpr, tpr)
    
    plt.figure()
    plt.plot(fpr, tpr, color='darkorange', lw=2, label=f'ROC curve (area = {roc_auc:.2f})')
    plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--')
    plt.xlim([0.0, 1.0])
    plt.ylim([0.0, 1.05])
    plt.xlabel('False Positive Rate')
    plt.ylabel('True Positive Rate')
    plt.title('Receiver Operating Characteristic')
    plt.legend(loc="lower right")
    plt.show()

# 7. 绘图

In [ ]:
def plot_feature_importance(predictor, feature_names, update_status):
    importance = predictor.feature_importance(test_data.drop(columns=['label']))
    importance_df = pd.DataFrame({'Feature': feature_names, 'Importance': importance})
    importance_df = importance_df.sort_values(by='Importance', ascending=False)
    
    plt.figure(figsize=(10, 6))
    sns.barplot(x='Importance', y='Feature', data=importance_df)
    plt.title('Feature Importance')
    plt.show()

# 8. 保存模型

In [ ]:
def save_model(predictor, model_path, update_status):
    predictor.save(model_path)
    update_status(f"Model saved to {model_path}")

# 9. 加载模型

In [ ]:
def load_model(model_path, update_status):
    predictor = TabularPredictor.load(model_path)
    update_status(f"Model loaded from {model_path}")
    return predictor

# 10. 训练

In [ ]:
# 1. 加载数据
file_path = '/opt/data/eurusd.parquet'
data = load_data(file_path)
    
# 2. 数据预处理
X, y = preprocess_data(data)
    
# 3. 处理正负样本数量差距过大
pos_weight = handle_imbalanced_data(y)
    
# 4. 特征工程
X_engineered = feature_engineering(X)
    
# 5. 划分训练集和测试集
X_train, X_test, y_train, y_test = train_test_split(X_engineered, y, test_size=0.2, random_state=42, stratify=y)
    
if args.load_path:
    # 加载预训练模型
    predictor = load_model(args.load_path, update_status)
else:
    # 使用AutoGluon进行超参数调优和模型训练
    predictor = train_with_autogluon(X_train, y_train, X_test, y_test, update_status)
        
    # 保存模型
    save_model(predictor, args.save_path, update_status)
    
# 模型评估
evaluate_model(predictor, X_test, y_test, update_status)
    
# 绘制特征重要性
plot_feature_importance(predictor, data.columns[:-1], update_status)

# 11 缠论label

In [1]:
from Chan import CChan
from ChanConfig import CChanConfig
from ChanModel.Features import CFeatures
from Common.CEnum import AUTYPE, DATA_SRC, KL_TYPE
from Common.CTime import CTime
from Plot.PlotDriver import CPlotDriver
import json
from typing import Dict, TypedDict

In [8]:
code = "eurusd"
begin_time = "2020-01-01"
end_time = "2020-03-01" 
data_src = DATA_SRC.PD
lv_list = [KL_TYPE.K_1M]

In [9]:
config = CChanConfig({
    "trigger_step": True,  # 打开开关！
    "bi_strict": True,
    "skip_step": 0,
    "divergence_rate": float("inf"),
    "bsp2_follow_1": False,
    "bsp3_follow_1": False,
    "min_zs_cnt": 0,
    "bs1_peak": False,
    "macd_algo": "peak",
    "bs_type": '1,2,3a,1p,2s,3b',
    "print_warning": True,
    "zs_algo": "normal",
})

chan = CChan(
    code=code,
    begin_time=begin_time,
    end_time=end_time,
    data_src=data_src,
    lv_list=lv_list,
    config=config,
    autype=AUTYPE.QFQ,
)

In [10]:
def plot(chan, plot_marker):
    plot_config = {
        "plot_kline": True,
        "plot_bi": True,
        "plot_seg": True,
        "plot_zs": True,
        "plot_bsp": True,
        "plot_marker": True,
    }
    plot_para = {
        "figure": {
            "x_range": 400,
        },
        "marker": {
            "markers": plot_marker
        }
    }
    plot_driver = CPlotDriver(
        chan,
        plot_config=plot_config,
        plot_para=plot_para,
    )
    plot_driver.save2img("label.png")

In [11]:
class T_SAMPLE_INFO(TypedDict):
    feature: CFeatures
    is_buy: bool
    open_time: CTime

bsp_dict: Dict[int, T_SAMPLE_INFO] = {}  # 存储策略产出的bsp的特征

def stragety_feature(last_klu):
    return {
        "open_klu_rate": (last_klu.close - last_klu.open)/last_klu.open,
    }

In [12]:
# 跑策略，保存买卖点的特征
for chan_snapshot in chan.step_load():
    last_klu = chan_snapshot[0][-1][-1]
    bsp_list = chan_snapshot.get_bsp()
    if not bsp_list:
        continue
    last_bsp = bsp_list[-1]

    cur_lv_chan = chan_snapshot[0]
    if last_bsp.klu.idx not in bsp_dict and cur_lv_chan[-2].idx == last_bsp.klu.klc.idx:
        # 假如策略是：买卖点分形第三元素出现时交易
        bsp_dict[last_bsp.klu.idx] = {
            "feature": last_bsp.features,
            "is_buy": last_bsp.is_buy,
            "open_time": last_klu.time,
        }
        bsp_dict[last_bsp.klu.idx]['feature'].add_feat(stragety_feature(last_klu))  # 开仓K线特征
        print(last_bsp.klu.time, last_bsp.is_buy)
print("买卖点已经跑完")

2020/01/01 19:50 False
2020/01/01 20:05 False
2020/01/01 20:14 False
2020/01/01 20:20 False
2020/01/01 21:33 False
2020/01/01 21:53 False
2020/01/01 23:20 False
2020/01/01 23:32 True
2020/01/01 23:44 True
2020/01/01 23:58 True
2020/01/02 01:27 True
2020/01/02 01:31 True
2020/01/02 01:49 False
2020/01/02 01:52 False
2020/01/02 01:56 True
2020/01/02 02:05 False
2020/01/02 02:14 True
2020/01/02 02:18 True
2020/01/02 02:21 True
2020/01/02 02:31 True
2020/01/02 02:51 False
2020/01/02 02:53 False
2020/01/02 03:02 False
2020/01/02 03:04 False
2020/01/02 03:09 True
2020/01/02 03:15 False
2020/01/02 03:21 False
2020/01/02 03:32 True
2020/01/02 03:39 True
2020/01/02 03:47 False
2020/01/02 03:53 False
2020/01/02 04:00 False
2020/01/02 04:03 False
2020/01/02 04:09 True
2020/01/02 04:37 True
2020/01/02 04:51 True
2020/01/02 04:57 True
2020/01/02 05:09 True
2020/01/02 05:21 True
2020/01/02 05:30 True
2020/01/02 05:43 False
2020/01/02 05:59 False
2020/01/02 06:06 False
2020/01/02 06:09 False
2020/01/

In [ ]:
# 生成libsvm样本特征
bsp_academy = [bsp.klu.idx for bsp in chan.get_bsp()]
feature_meta = {}  # 特征meta
cur_feature_idx = 0
plot_marker = {}
fid = open("feature.libsvm", "w")
for bsp_klu_idx, feature_info in bsp_dict.items():
    label = int(bsp_klu_idx in bsp_academy)  # 以买卖点识别是否准确为label
    features = []  # List[(idx, value)]
    for feature_name, value in feature_info['feature'].items():
        if feature_name not in feature_meta:
            feature_meta[feature_name] = cur_feature_idx
            cur_feature_idx += 1
        features.append((feature_meta[feature_name], value))
    features.sort(key=lambda x: x[0])
    feature_str = " ".join([f"{idx}:{value}" for idx, value in features])
    fid.write(f"{label} {feature_str}\n")
    plot_marker[feature_info["open_time"].to_str()] = ("√" if label else "×", "down" if feature_info["is_buy"] else "up")
fid.close()

with open("feature.meta", "w") as fid:
    # meta保存下来，实盘预测时特征对齐用
    fid.write(json.dumps(feature_meta))

In [ ]:
# 画图检查label是否正确
plot(chan, plot_marker)